# Vista Victoria By-Laws — Advanced RAG Pipeline

Adapted from the DeepLearning.ai *Advanced RAG* course (Lesson 1).  
**Document**: Vista Victoria Strata Plan 105121 registered by-laws  
**LLM**: Claude Haiku (Anthropic) — for both RAG answering and Ragas evaluation  
**Embeddings**: Local `BAAI/bge-small-en-v1.5` — no external API needed

---
### What we'll build
| Stage | Technique | Why it matters for by-laws |
|---|---|---|
| 1 | **Basic RAG** | Baseline — splits doc into fixed chunks, retrieves top-k by similarity |
| 2 | **Sentence Window** | Indexes individual sentences for precise retrieval, but expands each hit to a 3-sentence window before answering — great for dense legal text |
| 3 | **Auto-merging** | Hierarchical chunks (2048→512→128 tokens); if enough leaf chunks from the same parent are retrieved, merges up to the full section |

### How we evaluate — the Ragas triad
| Metric | Question it answers |
|---|---|
| **Faithfulness** | Is every claim in the answer actually supported by the retrieved text? |
| **Answer Relevancy** | Does the answer actually address the question asked? |
| **Context Precision** | Are the retrieved chunks genuinely about the question? |

All three scores are 0–1; higher is better. No ground-truth answers required.

## Setup

```bash
pip install -r requirements.txt
```

Create a `.env` file in this directory:
```
ANTHROPIC_API_KEY=sk-ant-...
```

In [5]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file"
print("API key loaded.")

API key loaded.


## 1. Load the By-Laws Document

In [6]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./bylaws_clean.txt"]
).load_data()

print(f"Loaded {len(documents)} document(s)")
print(f"Total characters: {sum(len(d.text) for d in documents):,}")
print("\n--- First 500 chars ---")
print(documents[0].text[:500])

Loaded 1 document(s)
Total characters: 155,576

--- First 500 chars ---
PAGE 1
Strata Plan 105121
Vista Victoria
312 Victoria Road
GLADESVILLE NSW 2111
Registered By-Laws

PAGE 2
Residual Document Version 05
Lodger Details
Lodger Code
502700U
Name
JANE CRITTENDEN LAWYER
Address
PO BOX 4623
SYDNEY 2001
Lodg


In [7]:
from llama_index.core import Document

# Merge into a single Document so chunking strategies span the whole text
document = Document(text="\n\n".join(d.text for d in documents))
print(f"Merged document: {len(document.text):,} chars")

Merged document: 155,576 chars


## 2. Basic RAG Pipeline

**How it works**
1. Split the by-laws into fixed-size chunks (~1024 tokens, 20-token overlap)
2. Embed each chunk locally with `BAAI/bge-small-en-v1.5` and store in a vector index
3. At query time: embed the question → find the top-k most similar chunks → send them to Claude → get an answer

**Limitation for legal text**: a single by-law clause often spans multiple sentences. A fixed chunk boundary may cut the clause in half, losing key context.

In [8]:
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

llm = Anthropic(model="claude-haiku-4-5-20251001", temperature=0.1)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

print("LLM and embedding model ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LLM and embedding model ready.


In [9]:
from llama_index.core import VectorStoreIndex

basic_index = VectorStoreIndex.from_documents([document])
basic_engine = basic_index.as_query_engine()
print("Basic vector index built.")

Basic vector index built.


In [10]:
response = basic_engine.query("What are the rules about keeping pets in the building?")
print(str(response))

# Rules About Keeping Pets in the Building

## Permitted Animals

You may keep:
- Goldfish or similar fish in an indoor aquarium
- A guide dog if you are visually or hearing impaired

For any other types or numbers of animals, you must obtain consent from the executive committee.

## Restrictions on Dogs

The executive committee will not consent to keeping:
- Large size dogs (except in exceptional circumstances with prescribed conditions)
- Dogs that are vicious, aggressive, noisy, or difficult to control
- Dogs that are not registered under the Companion Animals Act 1998 (NSW)
- Dangerous dogs under the Companion Animals Act 1998 (NSW)

## Control Requirements

- Your animal must not wander onto another lot or common property
- When taking your animal onto common property (such as for transport in and out of the building), you must restrain it by leash or pet cage and control it at all times

## Conditions and Responsibilities

- The executive committee may impose conditions when givi

In [11]:
response = basic_engine.query("Do I need approval from the owners corporation to renovate my apartment?")
print(str(response))

Whether you need approval from the Owners Corporation to renovate your apartment depends on the scope of your renovation work.

If your renovation involves work that is confined to the interior of your lot and does not affect Common Property, you generally would not need approval. However, if your renovation involves any of the following, you must first obtain approval from the Owners Corporation:

- Leaving anything on Common Property
- Obstructing the use of Common Property
- Using any part of Common Property for your own purposes
- Erecting any structure on Common Property
- Attaching any item to Common Property
- Doing anything that might cause damage to Common Property
- Altering Common Property

Additionally, if your renovation involves building works or alterations, specific provisions apply that may require approval.

If your renovation includes installing fixtures that serve your lot (such as built-in appliances or structural elements), you are responsible for maintaining them

In [12]:
response = basic_engine.query("What noise restrictions apply and during what hours?")
print(str(response))

Based on the provided context, there are no specific noise restrictions mentioned with designated hours. 

The context does address noise transmission related to floor coverings. It requires that floor space and surrounding areas within a Lot be covered or treated to reduce appropriately and suitably noise transmission that might unreasonably disturb another Owner or Occupier. Additionally, when installing certain floor surfaces and approved insulation (other than carpet with approved underlay), acoustic consultants must verify that the installation satisfies requirements related to noise transmission.

However, the context does not specify particular times of day during which noise restrictions apply or provide decibel limits or other quantifiable noise standards with time-based restrictions.


## 3. Evaluation with Ragas

### How Ragas evaluation works

For each question we collect three things from the query engine:
- **question** — the input
- **answer** — Claude's response
- **contexts** — the actual chunks that were retrieved and fed to Claude (`response.source_nodes`)

Ragas then uses Claude Haiku *as the judge* to score each (question, answer, contexts) triple.  
Everything runs on the same `ANTHROPIC_API_KEY` — no other keys needed.

```
query_engine.query(q)
      │
      ├─ answer      ──► Ragas judges: does the answer address the question?
      │                                                 → answer_relevancy
      │
      └─ source_nodes ─► Ragas judges: are these chunks on-topic?    → context_precision
                         Ragas judges: does the answer stay within    → faithfulness
                                       what the chunks actually say?
```

In [13]:
# Questions a Vista Victoria owner would realistically ask
eval_questions = [
    "What are the rules about keeping pets in the building?",
    "What noise restrictions apply and during what hours?",
    "Do I need approval to make renovations or alterations to my lot?"
    # "What are the parking rules for owners and visitors?",
    # "Who is responsible for maintaining common property?",
    # "Can I put up a satellite dish or air-conditioning unit on the outside of my apartment?",
    # "What are the rules about rubbish and waste disposal?",
    # "Am I allowed to run a business from my apartment?",
]

for i, q in enumerate(eval_questions, 1):
    print(f"{i}. {q}")

1. What are the rules about keeping pets in the building?
2. What noise restrictions apply and during what hours?
3. Do I need approval to make renovations or alterations to my lot?


### 3a. Evaluate the Basic RAG Pipeline

In [14]:
from utils import evaluate_pipeline

basic_df = evaluate_pipeline(basic_engine, eval_questions)
print("=== Basic RAG — aggregate scores ===")
print(basic_df[["faithfulness", "answer_relevancy"]].mean().round(3))

=== Basic RAG — aggregate scores ===
faithfulness        0.933
answer_relevancy    0.767
dtype: float64


In [26]:
# Per-question breakdown
basic_df[["question", "faithfulness", "answer_relevancy"]]

,question,faithfulness,answer_relevancy
0,What are the rules about keeping pets in the b...,0.95,0.95
1,What noise restrictions apply and during what ...,0.90,0.40
2,Do I need approval to make renovations or alte...,0.95,0.95


## 4. Advanced RAG — Sentence Window Retrieval

**How it works**
1. Each *sentence* gets its own index node → embedding captures a precise meaning unit
2. Each node also stores a `window` of the 3 surrounding sentences as metadata
3. At query time: retrieve the most relevant sentences, then **replace each hit with its window** before passing to Claude

**Why this helps for by-laws**: a clause like *"Owners must not allow their pet to enter the pool area"* is one sentence — indexing it alone gives precise retrieval. But Claude needs the surrounding context (what pets are allowed, what the pool area rules are) to give a complete answer. The window provides that without bloating the index.

In [16]:
from utils import build_sentence_window_index, get_sentence_window_query_engine

sentence_index = build_sentence_window_index(
    document, llm, embed_model, save_dir="sentence_index"
)
sw_engine = get_sentence_window_query_engine(sentence_index)
print("Sentence-window index built.")

Sentence-window index built.


In [17]:
response = sw_engine.query("What are the rules about keeping pets in the building?")
print(str(response))

# Rules About Keeping Pets

## Permitted Animals

You may keep:
- Goldfish or similar fish in an indoor aquarium
- A guide dog if you are visually or hearing impaired

For any other types or numbers of animals, you must obtain consent from the executive committee.

## Restrictions on Dogs

The executive committee will not approve consent to keep:
- Large size dogs (except in exceptional circumstances with prescribed conditions)
- Dogs that are vicious, aggressive, noisy, or difficult to control
- Dogs that are not registered under the Companion Animals Act 1998 (NSW)
- Dangerous dogs under the Companion Animals Act 1998 (NSW)

## Controlling Your Animal

- Your animal must not wander onto another resident's lot or common property
- When transporting your animal through common property, you must restrain it (by leash or pet cage) and control it at all times

## Conditions and Responsibilities

- The executive committee may impose conditions when granting consent to keep an animal
- You 

In [18]:
response = sw_engine.query("What noise restrictions apply and during what hours?")
print(str(response))

# Noise Restrictions

Based on the by-laws provided, there are noise restrictions related to floor coverings and installations in residential lots.

## Floor Covering Noise Requirements

You must ensure that all floor space and surrounding areas within your lot are covered or treated to reduce appropriately and suitably noise transmission that might unreasonably disturb another owner or occupier. This applies particularly when:

- Installing a floor surface and approved insulation (other than carpet with approved underlay)
- Removing or interfering with floor coverings or treatments

The by-laws require that floor installations meet a minimum sound proof rating as determined by the Owners Corporation's guidelines, and may require verification by qualified acoustic consultants.

## Important Note on Ongoing Obligations

Meeting the prescribed acoustic standards does not excuse you from an ongoing obligation to reduce appropriately and suitably noise transmission that might unreasonably 

### 4a. Evaluate the Sentence Window Pipeline

In [19]:
sw_df = evaluate_pipeline(sw_engine, eval_questions)
print("=== Sentence Window — aggregate scores ===")
print(sw_df[["faithfulness", "answer_relevancy"]].mean().round(3))

=== Sentence Window — aggregate scores ===
faithfulness        0.917
answer_relevancy    0.700
dtype: float64


In [20]:
sw_df[["question", "faithfulness", "answer_relevancy"]]

,question,faithfulness,answer_relevancy
0,What are the rules about keeping pets in the b...,0.95,0.95
1,What noise restrictions apply and during what ...,0.85,0.20
2,Do I need approval to make renovations or alte...,0.95,0.95


## 5. Advanced RAG — Auto-merging Retrieval

**How it works**
1. The document is split into a *hierarchy*: 2048-token parents → 512-token children → 128-token leaf chunks
2. Only the **leaf** (128-token) chunks are stored in the vector index — small enough for precise embedding
3. At query time: retrieve the most relevant leaves. If enough sibling leaves from the same 512-token parent are hit, **merge up** to the parent. Check again at the 2048 level.
4. Result: scattered retrievals automatically merge back into coherent by-law sections

**Why this helps for by-laws**: By-law 8 (Building Works) spans several pages with many sub-clauses. If your question about renovations triggers multiple sub-clause matches, auto-merging pulls the whole section rather than disconnected fragments.

In [21]:
from utils import build_automerging_index, get_automerging_query_engine

automerging_index = build_automerging_index(
    documents, llm, embed_model, save_dir="merging_index"
)
am_engine = get_automerging_query_engine(automerging_index)
print("Auto-merging index built.")

Auto-merging index built.


In [22]:
response = am_engine.query("What approval do I need to make alterations to my lot?")
print(str(response))

# Approvals Required for Alterations to Your Lot

The approval requirements depend on the type of alteration you're planning:

## Fire Doors
If you want to replace or make alterations to a fire door that gives access to your Lot, you must obtain **written approval from the Owners Corporation** before proceeding. Any alterations must also comply with fire regulations under the Building Code of Australia.

## Floor Coverings
If you plan to remove or interfere with floor coverings or treatments in your Lot, or install a floor surface with approved insulation other than carpet with approved underlay, you must obtain **written consent** before doing so.

## General Renovations
For renovations to your Lot that don't involve permanent changes to Common Property but are likely to inconvenience other Owners or Occupiers during the work, specific procedures apply under the renovation by-laws.

## Maintenance and Repairs
For installations or alterations that service your Lot (whether you made the

In [23]:
response = am_engine.query("What are the rules about keeping pets in the building?")
print(str(response))

> Merging 4 nodes into parent node.
> Parent node id: 0b7fcf2c-f023-4297-8a8c-1e290401eb9d.
> Parent node text: 13.3
The executive committee will not give you consent to keep any:
(a)
large size dog except ...

# Rules About Keeping Pets in the Building

## Permitted Animals

You may keep:
- Goldfish or other similar fish in an indoor aquarium
- A guide dog if you are visually or hearing impaired

## Consent Requirements

For any other types or numbers of animals, you must obtain consent from the executive committee.

## Restrictions on Dogs

The executive committee will not consent to keeping:
- Large size dogs (except in exceptional circumstances with prescribed conditions)
- Dogs that are vicious, aggressive, noisy, or difficult to control
- Dogs not registered under the Companion Animals Act 1998 (NSW)
- Dangerous dogs under the Companion Animals Act 1998 (NSW)

## Control and Containment

- Your animal must not wander onto another lot or common property
- When taking your animal o

### 5a. Evaluate the Auto-merging Pipeline

In [25]:
am_df = evaluate_pipeline(am_engine, eval_questions)
print("=== Auto-merging — aggregate scores ===")
print(am_df[["faithfulness", "answer_relevancy"]].mean().round(3))

> Merging 4 nodes into parent node.
> Parent node id: 0b7fcf2c-f023-4297-8a8c-1e290401eb9d.
> Parent node text: 13.3
The executive committee will not give you consent to keep any:
(a)
large size dog except ...

=== Auto-merging — aggregate scores ===
faithfulness        0.917
answer_relevancy    0.917
dtype: float64


In [ ]:
am_df[["question", "faithfulness", "answer_relevancy"]]

## 6. Side-by-side Comparison

In [ ]:
import pandas as pd

metrics = ["faithfulness", "answer_relevancy"]

comparison = pd.DataFrame(
    {
        "Basic RAG":       basic_df[metrics].mean(),
        "Sentence Window": sw_df[metrics].mean(),
        "Auto-merging":    am_df[metrics].mean(),
    }
).round(3)

print(comparison.to_string())
print("
Best per metric:")
print(comparison.idxmax(axis=1))

In [ ]:
# Per-question faithfulness comparison — useful for spotting which questions
# expose retrieval failures in each approach
faith_compare = pd.DataFrame({
    "question":        basic_df["question"],
    "basic":           basic_df["faithfulness"].round(3),
    "sentence_window": sw_df["faithfulness"].round(3),
    "auto_merging":    am_df["faithfulness"].round(3),
})
faith_compare

## Tips & Next Steps

### Reading Ragas scores (0–1)
| Score | Meaning |
|---|---|
| > 0.8 | Good |
| 0.5–0.8 | Moderate — worth investigating |
| < 0.5 | Poor — retrieval or generation problem |

### Common failure modes in legal text
- **Low context precision** → chunks are on a nearby topic but not the exact by-law  
  → Try increasing `similarity_top_k` and letting the reranker filter harder
- **Low faithfulness** → Claude is adding interpretation beyond the retrieved text  
  → Tighten the prompt (see custom prompt below)
- **Low answer relevancy** → the question is too vague; the retriever finds related but off-topic chunks  
  → Be more specific, e.g. *"By-law 12 — what does it say about noise?"*

### Switching to a more capable model for complex questions
```python
# Haiku — fast and cheap for exploration
llm = Anthropic(model="claude-haiku-4-5-20251001", temperature=0.1)

# Sonnet — better reasoning for multi-clause legal questions
llm = Anthropic(model="claude-sonnet-4-6", temperature=0.1)
```

### Adding a custom system prompt to reduce hallucination
```python
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    "You are a strata by-law assistant for Vista Victoria (SP105121), Gladesville NSW.\n"
    "Answer only from the context below. Cite the by-law number where applicable.\n"
    "If the context does not cover the question, say so explicitly — do not guess.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)
basic_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt})
```

In [27]:
q = "What are the rules about keeping pets?"
print("BASIC:"); print(basic_engine.query(q))
print("SENTENCE WINDOW:"); print(sw_engine.query(q))
print("AUTO-MERGING:"); print(am_engine.query(q))


BASIC:
# Rules About Keeping Pets

**Permitted Animals:**
You may keep goldfish or other similar fish in an indoor aquarium, or a guide dog if you are visually or hearing impaired. For any other types or numbers of animals, you must obtain consent from the executive committee.

**Restrictions on Dogs:**
The executive committee will not consent to keeping:
- Large size dogs (except in exceptional circumstances with prescribed conditions)
- Dogs that are vicious, aggressive, noisy, or difficult to control
- Dogs that are not registered under the Companion Animals Act 1998 (NSW)
- Dangerous dogs under the Companion Animals Act 1998 (NSW)

**Controlling Your Animal:**
- Your animal must not wander onto another lot or common property
- When taking your animal onto common property (such as transporting it in and out of the building), you must restrain it by leash or pet cage and control it at all times

**Conditions and Responsibilities:**
- The executive committee may impose conditions on p

In [28]:
response = sw_engine.query("What are the noise rules?")
print(str(response))
print("\n--- Retrieved chunks ---")
for i, node in enumerate(response.source_nodes, 1):
    print(f"\nChunk {i}:")
    print(node.text)


# Noise Rules

The noise rules primarily relate to floor coverings and installations:

## Floor Installation Noise Requirements

When installing floor surfaces and approved insulation (other than carpet with approved underlay), you must:

- Treat all floor space and surrounding areas within your lot to reduce noise transmission that might unreasonably disturb other owners or occupiers
- Comply with guidelines that may specify a minimum sound proof rating that the floor surface and surrounding areas must attain
- Engage qualified acoustic consultants to carry out acoustic services for non-carpet floor installations
- Provide confirmation within 14 days of installation, which may include a written report from a qualified acoustic consultant guaranteeing that the installation satisfies noise-related requirements

## Ongoing Obligation

Even if your installation meets the prescribed acoustic standards, you remain under an ongoing obligation to reduce noise transmission appropriately and su

In [29]:
nodes = list(basic_index.docstore.docs.values())
print(f"Total chunks: {len(nodes)}")
print(f"\n--- Chunk 0 ---")
print(nodes[0].text)
print(f"\n--- Chunk 1 ---")
print(nodes[1].text)


Total chunks: 40

--- Chunk 0 ---
PAGE 1
Strata Plan 105121
Vista Victoria
312 Victoria Road
GLADESVILLE NSW 2111
Registered By-Laws

PAGE 2
Residual Document Version 05
Lodger Details
Lodger Code
502700U
Name
JANE CRITTENDEN LAWYER
Address
PO BOX 4623
SYDNEY 2001
Lodger Box
1W
Email
JANE@JANECRITTENDENLAWYER.COM.AU
Reference
7297
Land Registry Document Identification
AV373804
STAMP DUTY:
Consolidation/Change of By-laws
Jurisdiction
NEW SOUTH WALES
Privacy Collection Statement
The information in this form is collected under statutory authority and used for the purpose of maintaining publicly searchable registers and
indexes.
Land Title Reference
CP/SP105121
N
Part Land Affected?
Land Description
Owners Corporation
THE OWNERS - STRATA PLAN NO. SP105121
Other legal entity
Meeting Date
05/08/2025
Amended by-law No.
Details
Added by-law No.
Details
Repealed by-law No.
Details
NOT APPLICABLE
SPECIAL BY-LAWS 1 and 2
NOT APPLICABLE
The subscriber requests the Registrar-General to make any nec

In [30]:
sample_text = nodes[0].text
embedding = embed_model.get_text_embedding(sample_text)
print(f"Dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")


Dimensions: 384
First 10 values: [-0.03537818789482117, 0.027262406423687935, 0.029369348660111427, 0.02158583700656891, 0.03288840129971504, 0.0431324765086174, -0.006981388665735722, 0.023205827921628952, -0.04022587463259697, 0.02643793635070324]


In [33]:
import numpy as np

vector_data = basic_index.vector_store._data
node_ids = list(vector_data.embedding_dict.keys())
print(f"Vectors stored: {len(node_ids)}")
first_vec = vector_data.embedding_dict[node_ids[0]]
print(f"Vector length: {len(first_vec)}")
print(f"First 10 values: {first_vec[:10]}")


def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

vec0 = vector_data.embedding_dict[node_ids[0]]
vec1 = vector_data.embedding_dict[node_ids[1]]
vec2 = vector_data.embedding_dict[node_ids[2]]

print(f"Chunk 0 vs Chunk 1: {cosine_similarity(vec0, vec1):.4f}")
print(f"Chunk 0 vs Chunk 2: {cosine_similarity(vec0, vec2):.4f}")
print(f"\nChunk 0 text: {nodes[0].text[:100]}")
print(f"Chunk 1 text: {nodes[1].text[:100]}")
print(f"Chunk 2 text: {nodes[2].text[:100]}")


Vectors stored: 40
Vector length: 384
First 10 values: [-0.03537818789482117, 0.027262406423687935, 0.029369348660111427, 0.02158583700656891, 0.03288840129971504, 0.0431324765086174, -0.006981388665735722, 0.023205827921628952, -0.04022587463259697, 0.02643793635070324]
Chunk 0 vs Chunk 1: 0.8074
Chunk 0 vs Chunk 2: 0.7559

Chunk 0 text: ============================================================
PAGE 1
Chunk 1 text: 105121 was affixed on
the 28 day of August 2025 in the presence of
Signature :.
Amy Robinson
-35
Chunk 2 text: hanging pictures or attaching items to those
walls).
by-laws are the by-laws under the Act in forc


In [35]:
# Auto-merging leaf nodes — should be much shorter
from llama_index.core.node_parser import get_leaf_nodes
am_nodes = list(automerging_index.docstore.docs.values())
leaf_nodes = [n for n in am_nodes if n.metadata.get("is_leaf", True)]
print(nodes[0].text[:300])         # basic chunk
print("---")
print(am_nodes[0].text[:300])      # auto-merging chunk


PAGE 1
Strata Plan 105121
Vista Victoria
312 Victoria Road
GLADESVILLE NSW 2111
Registered By-Laws

PAGE 2
---
PAGE 1
Strata Plan 105121
Vista Victoria
312 Victoria Road
GLADESVILLE NSW 2111
Registered By-Laws

PAGE 2


In [36]:
for i in range(3, 8):
    print(f"\n--- AM node {i} ({len(am_nodes[i].text)} chars) ---")
    print(am_nodes[i].text[:300])
    print(f"\n--- Basic node {i} ({len(nodes[i].text)} chars) ---")
    print(nodes[i].text[:300])
    print("="*60)



--- AM node 3 (9489 chars) ---
6.2
You must
(a)
comply with the Guidelines;
(b)
ensure that all floor space and surrounding areas within your Lot are
covered or otherwise treated in accordance with the Guidelines to
reduce appropriately and suitably noise transmission that might
Page 18 of 146


--- Basic node 3 (4123 chars) ---
Lot means a lot in the Strata Plan, being a Residential Lot, Commercial Lot or
Carspace Lot in Vista Victoria Gladesvilie and any Lots into which they are
subdivided or resubdivided.
O&M Manual means the operations and maintenance and manual for the
Building (as amended from time to time).
Occu

--- AM node 4 (9820 chars) ---
8.4
If you are an Owner, you must not carry out Building Work unless first you:
(a)
obtain a Checklist from the Building Manager;
(b)
submit plans detailing the proposed Building Work (including details of
tradespersons and contractors, materials, style, design, colour
schemes and any other d

--- Basic node 4 (4771 chars) ---
Signage Equ

In [ ]:
q = "What are the rules about floor coverings?"

for name, engine in [("Basic", basic_engine), ("Sentence Window", sw_engine), ("Auto-merging", am_engine)]:
    response = engine.query(q)
    print(f"\n{'='*40}")
    print(f"{name} answer:")
    print(str(response))



In [ ]:

q = "What are all my obligations as a lot owner regarding common property?"

#basic_engine.query(q).response

#sw_engine.query(q).response

am_engine.query(q).response

"# Lot Owner Obligations Regarding Common Property\n\nAs a lot owner, you have the following obligations regarding common property:\n\n**Access and Maintenance:**\n- You must give the Owners Corporation (or persons authorized by it) access to your lot according to notice provided, at your own cost\n- You must pay the Owners Corporation for its costs in doing work on your lot\n- You may be required to provide access to storage spaces within your lot for inspecting, testing, and maintaining services and infrastructure located within the common property (with prior notice, except in emergencies)\n\n**Damage and Hazards:**\n- You must not do anything or permit any authorized person to do anything on your lot or common property that is likely to create a hazard or danger to other lot owners, occupiers, or persons lawfully using the common property\n- If you or an authorized person causes damage to common property while using it, you must promptly notify the Owners Corporation and compensate

In [51]:
import textwrap

q = "What is the full process for getting approval to renovate my apartment?"

for name, engine in [("Basic", basic_engine), ("Sentence Window", sw_engine), ("Auto-merging", am_engine)]:
    print(f"\n{'='*60}")
    print(f"{name}")
    print('='*60)
    print(textwrap.fill(engine.query(q).response, width=80))



Basic
# Process for Getting Approval to Renovate Your Apartment  To obtain approval
for renovations to your apartment, you must follow these steps:  ## Before
Starting Work  1. **Obtain a Checklist** from the Building Manager  2. **Secure
Necessary Consents**    - Get approval from the Owners Corporation    - Obtain
consents from all relevant Government Agencies    - Provide written evidence of
all Government Agency consents to the executive committee  3. **Locate Service
Lines and Pipes**    - Find out where all service lines and pipes are located
before commencing work  4. **Notify the Building Manager**    - Provide 14 days'
written notice describing the proposed work in detail  5. **Arrange Work
Details** (if the executive committee deems it necessary)    - Agree on a
suitable time and means of access to the building    - Confirm work hours, work
methods, and debris disposal arrangements    - Designate a supervisor who will
be responsible for the work and available in emergencies 